# GPT-2-large Embeddings — SLURM Batch Submission

Submits one GPU job per interval to `gpt2_embedding_worker.py`, then concatenates
results into a single aligned embedding array.

**Efficiency note:** the worker uses GPT-2's KV cache so only the new word tokens
are forwarded each step — roughly 100–200× faster than re-running the full context
from scratch for every word.  Semantics are identical to the original approach
(each word sees all preceding words in the interval as left context).

**Output:**
- Per-interval: `embeddings/per_interval/{interval_id}_gpt2_embeddings.npy` — `float16 (n_words, 49, 1280)`
- Final: `embeddings/{patient}_gpt2_embeddings.npy` — `float16 (total_words, 49, 1280)`, row-aligned with `{patient}_transcripts.csv`

In [17]:
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

In [18]:
# ── Config ─────────────────────────────────────────────────────────────────────
PATIENT = "YFA"
VAD_ROOT = Path(f"/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/{PATIENT}")

TRANSCRIPTS_CSV = VAD_ROOT / f"{PATIENT}_transcripts.csv"
EMBEDDINGS_ROOT = VAD_ROOT / "embeddings"
PER_INTERVAL_DIR = EMBEDDINGS_ROOT / "per_interval"
PER_INTERVAL_DIR.mkdir(parents=True, exist_ok=True)

# SLURM
PARTITION = "guppy"
QOS = "default_tier"
CPUS = 4
MEM = "16G"
# 5 MPS units ≈ 2.4 GB of the L40's 48 GB — comfortable for GPT-2-large in fp16
GRES = "mps:l40:5"

PYTHON_BIN = "/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3"
WORKER_PATH = Path("/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/generate_embeddings/gpt2_embedding_worker.py")

LOGS_DIR = VAD_ROOT / "gpt2_emb_slurm_logs"
SCRIPTS_DIR = VAD_ROOT / "gpt2_emb_slurm_scripts"
LOGS_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print("transcripts:", TRANSCRIPTS_CSV)
print("output dir: ", EMBEDDINGS_ROOT)

transcripts: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/YFA_transcripts.csv
output dir:  /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/embeddings


In [19]:
# ── Scan eligible intervals ────────────────────────────────────────────────────
transcripts = pd.read_csv(TRANSCRIPTS_CSV)
print(f"total words: {len(transcripts)}")

interval_counts = transcripts.groupby("interval_id").size().rename("n_words")
print(f"intervals:   {len(interval_counts)}")

status = []
for iid, n_words in interval_counts.items():
    done = (PER_INTERVAL_DIR / f"{iid}_SUCCESS").exists()
    error = (PER_INTERVAL_DIR / f"{iid}_error.txt").exists()
    status.append({"interval_id": iid, "n_words": n_words, "done": done, "error": error})

status_df = pd.DataFrame(status)
eligible = status_df[~status_df["done"]].copy()

print(f"done:        {status_df.done.sum()}")
print(f"errored:     {status_df.error.sum()}")
print(f"to submit:   {len(eligible)}")
eligible.head(10)

total words: 610928
intervals:   1030
done:        1030
errored:     9
to submit:   0


,interval_id,n_words,done,error


In [9]:
# ── Submit SLURM jobs ──────────────────────────────────────────────────────────
# SUBMIT_ONE = True to validate a single job before releasing the full batch.
DRY_RUN = False
SUBMIT_ONE = False   # flip to False once validated

submitted = []
skipped = []
failed_submissions = []

for _, row in eligible.iterrows():
    iid = row["interval_id"]

    if (PER_INTERVAL_DIR / f"{iid}_SUCCESS").exists():
        skipped.append(iid)
        continue

    sbatch_text = f"""#!/bin/bash
#SBATCH --job-name=gpt2_{iid}
#SBATCH --partition={PARTITION}
#SBATCH --qos={QOS}

#SBATCH --cpus-per-task={CPUS}
#SBATCH --mem={MEM}
#SBATCH --gres={GRES}
#SBATCH --output={LOGS_DIR}/{iid}_%j.out
#SBATCH --error={LOGS_DIR}/{iid}_%j.err

set -eo pipefail

PYTHON={PYTHON_BIN}

echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "INTERVAL: {iid}"

$PYTHON {WORKER_PATH} \\
    "{TRANSCRIPTS_CSV}" \\
    "{iid}" \\
    "{PER_INTERVAL_DIR}"

echo "END: $(date)"
"""

    sbatch_path = SCRIPTS_DIR / f"{iid}.sbatch"
    sbatch_path.write_text(sbatch_text)

    if DRY_RUN:
        print(f"dry-run: {iid}")
        submitted.append((iid, "dry-run"))
    else:
        try:
            res = subprocess.run(
                ["sbatch", str(sbatch_path)],
                capture_output=True, text=True, check=True,
            )
            submitted.append((iid, res.stdout.strip()))
            print(f"submitted: {iid} -> {res.stdout.strip()}")
        except subprocess.CalledProcessError as e:
            failed_submissions.append((iid, e.stderr.strip()))
            print(f"FAILED: {iid}\n{e.stderr}")

    if SUBMIT_ONE and not DRY_RUN and submitted:
        print("SUBMIT_ONE=True — stopping after first submission.")
        break

print(f"\nsubmitted: {len(submitted)}  skipped: {len(skipped)}  failed: {len(failed_submissions)}")

submitted: 20240409-122835_0000 -> Submitted batch job 219404
submitted: 20240409-122835_0001 -> Submitted batch job 219405
submitted: 20240409-122835_0002 -> Submitted batch job 219406
submitted: 20240409-122835_0009 -> Submitted batch job 219407
submitted: 20240409-135651_0000 -> Submitted batch job 219408
submitted: 20240409-135651_0001 -> Submitted batch job 219409
submitted: 20240409-135651_0002 -> Submitted batch job 219410
submitted: 20240409-135651_0003 -> Submitted batch job 219411
submitted: 20240409-135651_0004 -> Submitted batch job 219412
submitted: 20240409-135651_0005 -> Submitted batch job 219413
submitted: 20240409-135651_0006 -> Submitted batch job 219414
submitted: 20240409-135651_0007 -> Submitted batch job 219415
submitted: 20240409-135651_0008 -> Submitted batch job 219416
submitted: 20240409-135651_0009 -> Submitted batch job 219417
submitted: 20240409-135651_0010 -> Submitted batch job 219418
submitted: 20240409-135651_0011 -> Submitted batch job 219419
submitte

In [5]:
# ── Check first submission ─────────────────────────────────────────────────────
if submitted:
    iid, job_info = submitted[0]
    print(f"interval: {iid}")
    print(f"job:      {job_info}")
    print()
    success = (PER_INTERVAL_DIR / f"{iid}_SUCCESS").exists()
    error_path = PER_INTERVAL_DIR / f"{iid}_error.txt"
    print(f"_SUCCESS:  {success}")
    print(f"error.txt: {error_path.exists()}")
    if error_path.exists():
        print(error_path.read_text())
    if success:
        emb = np.load(str(PER_INTERVAL_DIR / f"{iid}_gpt2_embeddings.npy"))
        meta = json.loads((PER_INTERVAL_DIR / f"{iid}_meta.json").read_text())
        print(f"\nembedding shape: {emb.shape}  dtype: {emb.dtype}")
        print(f"n_words in meta: {meta['n_words']}")

interval: 20251217-095026_0126
job:      Submitted batch job 203239

_SUCCESS:  False
error.txt: True
Traceback (most recent call last):
  File "/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/generate_embeddings/gpt2_embedding_worker.py", line 122, in main
    tokenizer, model = build_model(device)
                       ^^^^^^^^^^^^^^^^^^^
  File "/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/generate_embeddings/gpt2_embedding_worker.py", line 37, in build_model
    model = model.to(device).eval()
            ^^^^^^^^^^^^^^^^
  File "/scratch/tahaismail424/miniforge3/envs/speech_247_env/lib/python3.11/site-packages/transformers/modeling_utils.py", line 4343, in to
    return super().to(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/scratch/tahaismail424/miniforge3/envs/speech_247_env/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1381, in to
    return self._apply(convert)
           ^^^^^^^^^^^^^^^^^^^^
  File "/scratch/tahaismail424

In [20]:
# ── Concatenate into full aligned embedding array ──────────────────────────────
# Run this cell after all jobs complete.
# Produces a (total_words, 37, 1280) float16 array row-aligned with transcripts.csv.
# GPT-2-large: 1 embedding layer + 36 transformer layers = 37 hidden states total.

transcripts = pd.read_csv(TRANSCRIPTS_CSV)

# Infer true shape from first available embedding file
n_layers, hidden_dim = 37, 1280
for iid in transcripts["interval_id"].unique():
    p = PER_INTERVAL_DIR / f"{iid}_gpt2_embeddings.npy"
    if p.exists():
        _e = np.load(str(p), mmap_mode="r")
        n_layers, hidden_dim = _e.shape[1], _e.shape[2]
        break

all_embeddings = np.full((len(transcripts), n_layers, hidden_dim), np.nan, dtype=np.float16)
loaded = 0
missing = []

for iid, group in transcripts.groupby("interval_id", sort=False):
    emb_path = PER_INTERVAL_DIR / f"{iid}_gpt2_embeddings.npy"
    if not emb_path.exists():
        missing.append(iid)
        continue
    emb = np.load(str(emb_path))   # (n_words_interval, n_layers, hidden_dim)
    if emb.shape[0] != len(group):
        print(f"WARNING {iid}: emb has {emb.shape[0]} rows but CSV has {len(group)} — skipping")
        missing.append(iid)
        continue
    all_embeddings[group.index] = emb
    loaded += 1

print(f"loaded: {loaded} / {transcripts.interval_id.nunique()} intervals")
if missing:
    print(f"missing ({len(missing)}): {missing[:10]}{'...' if len(missing) > 10 else ''}")
print(f"embedding shape: {all_embeddings.shape}  dtype: {all_embeddings.dtype}")
nan_rows = np.isnan(all_embeddings[:, 0, 0]).sum()
print(f"nan rows: {nan_rows} / {len(transcripts)}")

out_path = EMBEDDINGS_ROOT / f"{PATIENT}_gpt2_embeddings.npy"
np.save(str(out_path), all_embeddings)
print(f"saved to {out_path}")

loaded: 1030 / 1030 intervals
embedding shape: (610928, 37, 1280)  dtype: float16
nan rows: 0 / 610928
saved to /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/embeddings/YFA_gpt2_embeddings.npy
